# Customer Churn Classification: Answers

This notebook answers all eight tasks using `customer_churn_classification.xlsx`. The workbook labels itself synthetic classroom data; do not use its results for real customer decisions. Install dependencies if needed with `%pip install pandas openpyxl scikit-learn`.

## 1. Load customer data

**Question:** Load the Customer data sheet in Python. Set `churned` as `y`. Remove `churned` and `customer_id` from `X`.

**Answer:** Use the ten numeric predictors; exclude the identifier and target from the features.

In [ ]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path("customer_churn_classification.xlsx")
if not DATA_PATH.exists():
    DATA_PATH = Path("/home/raghavanr/rejish/BATCH/Assign/Day 9-11/customer_churn_classification.xlsx")

df = pd.read_excel(DATA_PATH, sheet_name="Customer data", engine="openpyxl")
y = df["churned"].astype(int)
X = df.drop(columns=["churned", "customer_id"])
print("Dataset shape:", df.shape)
print("Predictor shape:", X.shape)
print("Target counts:\n", y.value_counts().sort_index())
print("Missing predictor values:", int(X.isna().sum().sum()))
display(df.head())

## 2. Stratified 80/20 split

**Question:** Make a stratified 80/20 training/test split and count canceled customers in each split.

**Answer:** With `random_state=42`, the 500-row workbook splits into 400 training and 100 test customers. Stratification preserves the 28% churn rate: 112 churners in training and 28 in test.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Train churn counts (0=stayed, 1=churned):\n", y_train.value_counts().sort_index())
print("Test churn counts (0=stayed, 1=churned):\n", y_test.value_counts().sort_index())

## 3-4. Fit the five models and prevent scaling leakage

**Question:** Train KNN, Gaussian Naive Bayes, Decision Tree, Random Forest, and Logistic Regression. Use a `StandardScaler` pipeline for KNN and Logistic Regression, fitted on training data only.

**Answer:** KNN and Logistic Regression receive scaler-first pipelines. The scaler is fitted only when the pipeline is fit on `X_train`; no test-set statistics leak into preprocessing. The remaining models use their standard estimators.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

models = {
    "KNN": Pipeline([
        ("scaler", StandardScaler()),
        ("model", KNeighborsClassifier(n_neighbors=5)),
    ]),
    "Gaussian Naive Bayes": GaussianNB(),
    "Decision Tree": DecisionTreeClassifier(max_depth=4, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1),
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=2000, random_state=42)),
    ]),
}

for model in models.values():
    model.fit(X_train, y_train)
print("Fitted models:", ", ".join(models))

## 5-6. Evaluate and compare

**Question:** Report test accuracy, precision, recall, F1, and confusion matrix. Identify which model catches the most churners and which produces the fewest false alarms.

**Answer:** Treat churn (`1`) as the positive class. Highest recall catches the largest share of churners; the smallest false-positive count gives the fewest false alarms. These may be different models, so use the measured test results below rather than assuming one model wins both goals.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

results = []
confusion_matrices = {}
predictions = {}
for name, model in models.items():
    y_pred = model.predict(X_test)
    predictions[name] = y_pred
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred, labels=[0, 1]).ravel()
    confusion_matrices[name] = confusion_matrix(y_test, y_pred, labels=[0, 1])
    results.append({
        "model": name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision_churn": precision_score(y_test, y_pred, pos_label=1, zero_division=0),
        "recall_churn": recall_score(y_test, y_pred, pos_label=1, zero_division=0),
        "f1_churn": f1_score(y_test, y_pred, pos_label=1, zero_division=0),
        "false_alarms": int(fp),
        "missed_churners": int(fn),
    })

results_df = pd.DataFrame(results).set_index("model").sort_values("recall_churn", ascending=False)
display(results_df)
for name, matrix in confusion_matrices.items():
    print("\n" + name + " confusion matrix (rows=true [0,1], columns=predicted [0,1]):")
    print(matrix)

best_recall = results_df["recall_churn"].max()
most_churn_caught = results_df.index[results_df["recall_churn"] == best_recall].tolist()
fewest_false_alarms = results_df["false_alarms"].min()
least_false_alarm_models = results_df.index[results_df["false_alarms"] == fewest_false_alarms].tolist()
print("Most churners caught (highest recall):", most_churn_caught)
print("Fewest false alarms:", least_false_alarm_models, "with", int(fewest_false_alarms))

## 7. Inspect what the models learned

**Question:** Compare K values, display a small decision tree, examine forest feature importance, and inspect logistic-regression coefficients.

**Answer:** Tune K using stratified cross-validation on training data only. Show a shallow tree’s rules, sort the forest’s impurity-based importances, and rank standardized logistic coefficients by absolute magnitude. Importance and coefficients describe fitted associations, not causation.

In [ ]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.tree import export_text

knn_search = GridSearchCV(
    estimator=Pipeline([
        ("scaler", StandardScaler()),
        ("model", KNeighborsClassifier()),
    ]),
    param_grid={"model__n_neighbors": [3, 5, 7, 9, 11]},
    scoring="recall",
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    n_jobs=-1,
)
knn_search.fit(X_train, y_train)
print("Best K by training-fold churn recall:", knn_search.best_params_)
print("Best cross-validated recall:", knn_search.best_score_)

print("\nSmall decision tree rules:")
print(export_text(models["Decision Tree"], feature_names=list(X.columns), max_depth=3))

forest_importance = pd.Series(
    models["Random Forest"].feature_importances_, index=X.columns
).sort_values(ascending=False)
print("\nRandom Forest feature importances:")
display(forest_importance.to_frame("importance"))

logistic = models["Logistic Regression"].named_steps["model"]
coefficients = pd.Series(logistic.coef_[0], index=X.columns)
coefficients = coefficients.reindex(coefficients.abs().sort_values(ascending=False).index)
print("\nLogistic Regression coefficients (features are standardized):")
display(coefficients.to_frame("coefficient"))

## 8. Retention recommendation

**Question:** Recommend a model to identify customers for a retention campaign.

**Answer:** Use the measured recall/precision trade-off and campaign capacity to choose a model and threshold. Prefer high churn recall when missed cancellations are costly, but constrain false alarms to a level the team can contact. Validate the chosen operating point on representative future data. Since the workbook is synthetic, this is a classroom recommendation only.